#Competitive Threat and Market Share Loss Prediction

##Libraries

In [ ]:
# if kagglehub not found
!pip install --upgrade kagglehub

In [8]:
import kagglehub
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import nltk

In [2]:
news_path = kagglehub.dataset_download("miguelaenlle/massive-stock-news-analysis-db-for-nlpbacktests")

print("Path to dataset files:", news_path)

100%|██████████| 210M/210M [00:04<00:00, 54.6MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/miguelaenlle/massive-stock-news-analysis-db-for-nlpbacktests/versions/2


In [ ]:
stock_path = kagglehub.dataset_download("ehallmar/daily-historical-stock-prices-1970-2018")

print("Path to dataset files:", stock_path)

100%|██████████| 469M/469M [00:04<00:00, 116MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/ehallmar/daily-historical-stock-prices-1970-2018/versions/1


##Dataset

In [6]:
# news headlines
dataset_root = os.path.join(news_path, 'analyst_ratings_processed.csv')
news = pd.read_csv(dataset_root)
news

,Unnamed: 0,title,date,stock
0,0.0,Stocks That Hit 52-Week Highs On Friday,2020-06-05 10:30:00-04:00,A
1,1.0,Stocks That Hit 52-Week Highs On Wednesday,2020-06-03 10:45:00-04:00,A
2,2.0,71 Biggest Movers From Friday,2020-05-26 04:30:00-04:00,A
3,3.0,46 Stocks Moving In Friday's Mid-Day Session,2020-05-22 12:45:00-04:00,A
4,4.0,B of A Securities Maintains Neutral on Agilent...,2020-05-22 11:38:00-04:00,A
...,...,...,...,...
1400464,1413844.0,Top Narrow Based Indexes For August 29,2011-08-29 10:41:00-04:00,ZX
1400465,1413845.0,Recap: Wednesday's Top Percentage Gainers and ...,2011-06-22 16:44:00-04:00,ZX
1400466,1413846.0,UPDATE: Oppenheimer Color on China Zenix Auto ...,2011-06-21 08:26:00-04:00,ZX
1400467,1413847.0,Oppenheimer Initiates China Zenix At Outperfor...,2011-06-21 05:59:00-04:00,ZX


In [ ]:
# historic stock prices
dataset_root = os.path.join(stock_path, 'historical_stock_prices.csv')
stock = pd.read_csv(dataset_root)
stock

,ticker,open,close,adj_close,low,high,volume,date
0,AHH,11.50,11.58,8.493155,11.25,11.68,4633900,2013-05-08
1,AHH,11.66,11.55,8.471151,11.50,11.66,275800,2013-05-09
2,AHH,11.55,11.60,8.507822,11.50,11.60,277100,2013-05-10
3,AHH,11.63,11.65,8.544494,11.55,11.65,147400,2013-05-13
4,AHH,11.60,11.53,8.456484,11.50,11.60,184100,2013-05-14
...,...,...,...,...,...,...,...,...
20973884,NZF,14.60,14.59,14.590000,14.58,14.62,137500,2018-08-20
20973885,NZF,14.60,14.58,14.580000,14.57,14.61,151200,2018-08-21
20973886,NZF,14.58,14.59,14.590000,14.57,14.63,185400,2018-08-22
20973887,NZF,14.60,14.57,14.570000,14.57,14.64,135600,2018-08-23


###Data Cleaning & Pre-processing

In [ ]:
# remove "unnameed: 0" column from news_df, duplicates as the index


In [ ]:
# reuncate "date" info to contain only YYYY-MM-DD from news_df, so take out the time


In [ ]:
# remove any rows with null values from news_df


In [ ]:
# remove any rows from news_df where the date exceeds date range of stock_df


In [ ]:
# remove rows from news_df where value in "stock" does not match "ticker" value in stock_df


In [ ]:
# remove any rows with null values from stock_df


In [ ]:
# remove any rows from stock_df where the date exceeds the date range of news_df


In [ ]:
# remove rows from stock_df where value in "ticker" value does not match "stock" value in news_df


In [ ]:
# by matching the date of the news headline and the stock ticker,
# add the prices of the stock from the past 3 days to new columns in news_df
## ex. news_date(2025-04-23); add values from stock_df with dates {2025-04-21,2025-04-22, 2025-04-23} to 3 new columns


In [ ]:
# by matching the date of the news headline and the stock ticker,
# add the next day's stock price to the news_df as a new column
## ex. news_date(2025-04-23); add values from stock_df with dates {2025-04-24} to a new column 'next_day'


In [ ]:
# add column in news_df for whether stock prices went up or down
## ex. price from 1_day_ago is higher than price of next_day -> price_went_up column value = 1
## ex. price from 1_day_ago is lower than price of next_day -> price_went_up column value = 0


##Visualizations

In [ ]:
# plot the top # most mentioned stock ticker in news_df


In [ ]:
# plot the top # most traded stocks in stock_df


In [ ]:
# plot the top # stocks with the largest price changes within a day


In [ ]:
# plot counts of top bigrams (n-grams)


In [ ]:
# plot price_went_up column


In [ ]:
# other plots you deem fit


##FINBert 2.0

In [11]:
finbert = pipeline(
    "text-classification",
    model = "ProsusAI/finbert"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [13]:
news = news.head(3)
news

,Unnamed: 0,title,date,stock
0,0.0,Stocks That Hit 52-Week Highs On Friday,2020-06-05 10:30:00-04:00,A
1,1.0,Stocks That Hit 52-Week Highs On Wednesday,2020-06-03 10:45:00-04:00,A
2,2.0,71 Biggest Movers From Friday,2020-05-26 04:30:00-04:00,A


In [14]:
headlines = news['title'].tolist()

results = finbert(
    headlines,
    batch_size = 32,
    truncation = True
)

news['sentitment'] = [r['label'] for r in results]
news['confidence'] = [r['score'] for r in results]

news

/tmp/ipykernel_210/1846482169.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  news['sentitment'] = [r['label'] for r in results]
/tmp/ipykernel_210/1846482169.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  news['confidence'] = [r['score'] for r in results]


,Unnamed: 0,title,date,stock,sentitment,confidence
0,0.0,Stocks That Hit 52-Week Highs On Friday,2020-06-05 10:30:00-04:00,A,neutral,0.706543
1,1.0,Stocks That Hit 52-Week Highs On Wednesday,2020-06-03 10:45:00-04:00,A,neutral,0.668674
2,2.0,71 Biggest Movers From Friday,2020-05-26 04:30:00-04:00,A,neutral,0.796792


In [9]:

news["sentitment"] = news["title"].apply(
    lambda x: finbert(str(x))[0]['label']
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
news

##FINBert (NLP)

In [3]:
# Load pre-trained FINBERT model and tokenizer
tokenizer = AutoTokenizer.from_pretrained('ProsusAI/finbert')
model = AutoModelForSequenceClassification.from_pretrained('ProsusAI/finbert')

# Set model to evaluation mode
model.eval()

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [4]:
def get_sentiment(text):
    if pd.isna(text):
        return None

    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)

    # Get the predicted class (0: positive, 1: negative, 2: neutral)
    logits = outputs.logits
    probabilities = torch.softmax(logits, dim=1)

    # Map to sentiment labels
    sentiment_labels = ['positive', 'negative', 'neutral']
    predicted_class_idx = torch.argmax(probabilities, dim=1).item()

    return sentiment_labels[predicted_class_idx]

In [7]:
news['sentiment'] = news['title'].apply(get_sentiment)
display(news.head())

KeyboardInterrupt: 

##Data Spliting

In [ ]:
X = news.drop('price_went_up', axis=1)
y = news['price_went_up']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=6, stratify=True)

##Random Forest